### Ноутбук, в котором генерируются фичи с использованием OpenFE

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from ml_pipeline.config import config
from ml_pipeline.utils import set_seed
from ml_pipeline.pipeline import MLPipeline

In [2]:
set_seed(config.general.seed)

In [3]:
full_df = pd.read_csv(config.paths.train, index_col="PassengerId")
full_test_df = pd.read_csv(config.paths.test, index_col="PassengerId")
full_df.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [7]:
config_logreg = config.copy()

model_name = "logistic_regression"

config_logreg.experiment.to_train = [
    {
        "model": model_name
    }
]

config_logreg.split.cv = False

elastic_pipeline = MLPipeline(config_logreg)

elastic_pipeline.run(train_df=full_df)

0. logistic_regression
Prepared model with params: {'l1_ratio': 0, 'solver': 'lbfgs', 'max_iter': 500}
Single split accuracy: 0.8156
-> Best fold: 1, CV accuracy: 0.8156, std: 0.0000



In [8]:
from ml_pipeline.preprocessing import TRANSFORMER_REGISTRY

def _build_transformers(model_config: dict, config) -> list:
    """
    Собрать пошаговый список препроцессоров
    """
    transformers = []

    preprocessing_steps = []

    model_preprocessing_type = model_config.get("preprocessing")

    # если указан кастомный список препроцессоров, использовать его вместо дефолтного
    if model_preprocessing_type == "custom":
        preprocessing_steps = model_config.get("preprocessing_steps") or []
    else:
        preprocessing_steps = config.preprocessing.default

    registry = config.preprocessing.registry

    for step in preprocessing_steps:

        name = step["name"]

        if name not in TRANSFORMER_REGISTRY:
            raise ValueError(f"Неизвестный препроцессор '{name}' - нет в реестре")

        # дефолтные параметры препроцессора из реестра
        default_params = registry.get(name) or {}
        # кастомные параметры препроцессора из эксперимента
        custom_params = step.get(name) or {}

        merged = {}

        # влить конфиги вместе, кастомные параметры перезаписывают дефолтные при наличии
        if default_params or custom_params:
            merged = {**default_params, **custom_params}

        # передавать конфиг только если в нем есть параметры
        if len(merged.items()):
            transformers.append(TRANSFORMER_REGISTRY[name](**merged))
        else:
            transformers.append(TRANSFORMER_REGISTRY[name]())

    return transformers


def _apply_transformers(
    transformers: list, input_df: pd.DataFrame, fit: bool
) -> pd.DataFrame:
    """
    Применить (при необходимости фиттить) список препроцессоров
    """
    df = input_df.copy()

    for t in transformers:
        if fit:
            t.fit(df)
        df = t.transform(df)

    return df

In [11]:
model_config = config_logreg.models.get(model_name)

target = config_logreg.data.target_col

y = full_df[target].values
X_train_raw = full_df.drop(columns=[target])

preprocess_transformers = _build_transformers(model_config, config_logreg)

X_train = _apply_transformers(preprocess_transformers, X_train_raw, fit=True)
X_test = _apply_transformers(preprocess_transformers, full_test_df, fit=False)

In [14]:
from openfe import OpenFE, transform

ofe = OpenFE()

features = ofe.fit(data=X_train, label=y, n_jobs=4)

The number of candidate features is 1449
Start stage I selection.


100%|██████████| 16/16 [00:05<00:00,  2.90it/s]


1147 same features have been deleted.
Meet early-stopping in successive feature-wise halving.


100%|██████████| 16/16 [00:03<00:00,  4.59it/s]


The number of remaining candidate features is 301
Start stage II selection.


100%|██████████| 16/16 [00:02<00:00,  6.09it/s]


Finish data processing.
[LightGBM] [Info] Number of positive: 273, number of negative: 439
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002242 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7746
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 310


In [15]:
train_x, test_x = transform(X_train, X_test, features, n_jobs=4) # transform the train and test data according to generated features.

In [27]:
train_x.index.name = 'PassengerId'
train_x['Survived'] = y
train_x

,Pclass,Sex,SibSp,Parch,Family_Size,Alone,More_Than_4_relatives,Embarked_C,Embarked_S,Age_Group,...,autoFE_f_292,autoFE_f_293,autoFE_f_294,autoFE_f_295,autoFE_f_296,autoFE_f_297,autoFE_f_298,autoFE_f_299,autoFE_f_300,Survived
PassengerId,,,,,,,,,,,,,,,,,,,,,
1,3,0,1,0,1,0,0,0.0,1.0,2,...,1.0,0.722144,1.000000,0.846373,0.837485,359.0,2.0,3.0,0.420000,0
2,1,1,1,0,1,0,0,1.0,0.0,3,...,1.0,0.657895,1.000000,0.846373,0.920601,160.0,3.0,2.0,0.883721,1
3,3,1,0,0,0,1,0,0.0,1.0,2,...,1.0,0.333568,0.000000,0.361858,0.980687,557.0,2.0,4.0,0.420000,1
4,1,1,1,0,1,0,0,0.0,1.0,3,...,1.0,0.657895,1.000000,0.846373,0.051502,359.0,3.0,2.0,0.383721,1
5,3,0,0,0,0,1,0,0.0,1.0,3,...,1.0,0.333568,0.000000,0.361858,0.692764,557.0,3.0,3.0,0.383721,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
887,2,0,0,0,0,1,0,0.0,1.0,2,...,1.0,0.287004,0.000000,0.361858,0.155397,557.0,2.0,2.0,0.420000,0
888,1,1,0,0,0,1,0,0.0,1.0,2,...,1.0,0.249226,0.000000,0.361858,0.023605,557.0,2.0,2.0,0.420000,1
889,3,1,1,2,3,0,0,0.0,1.0,2,...,1.0,0.893512,1.732051,0.846373,0.992489,359.0,3.0,4.0,0.420000,0


In [28]:
test_x

,Pclass,Sex,SibSp,Parch,Family_Size,Alone,More_Than_4_relatives,Embarked_C,Embarked_S,Age_Group,...,autoFE_f_291,autoFE_f_292,autoFE_f_293,autoFE_f_294,autoFE_f_295,autoFE_f_296,autoFE_f_297,autoFE_f_298,autoFE_f_299,autoFE_f_300
PassengerId,,,,,,,,,,,,,,,,,,,,,
892,3,0,0,0,0,1,0,0.0,0.0,3,...,0.647094,1.0,0.333568,0.000000,0.361858,0.353499,233.0,3.0,3.0,0.383721
893,3,1,1,0,1,0,0,0.0,1.0,3,...,0.292918,1.0,0.722144,1.000000,0.846373,0.482833,359.0,3.0,4.0,0.383721
894,2,0,0,0,0,1,0,0.0,0.0,4,...,0.647094,1.0,0.287004,0.000000,0.361858,0.179715,233.0,4.0,2.0,0.349057
895,3,0,0,0,0,1,0,0.0,1.0,2,...,0.647094,1.0,0.333568,0.000000,0.361858,0.343416,557.0,2.0,3.0,0.420000
896,3,1,1,1,2,0,0,0.0,1.0,2,...,0.292918,1.0,0.830748,1.414214,0.846373,0.379828,359.0,2.0,4.0,0.420000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1305,3,0,0,0,0,1,0,0.0,1.0,3,...,0.647094,1.0,0.333568,0.000000,0.361858,0.689798,557.0,3.0,3.0,0.383721
1306,1,1,0,0,0,1,0,1.0,0.0,3,...,0.792918,1.0,0.249226,0.000000,0.361858,0.783262,233.0,3.0,2.0,0.883721
1307,3,0,0,0,0,1,0,0.0,1.0,3,...,0.647094,1.0,0.333568,0.000000,0.361858,0.795374,557.0,3.0,3.0,0.383721


In [29]:
train_x.to_csv(f"data/train_openfe.csv", index=True)

In [30]:
test_x.to_csv(f"data/test_openfe.csv", index=True)